In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')

!pip install music21
!pip install graphviz

#if not os.path.isdir("/content/drive/MyDrive/results"):
    #Path("/content/drive/MyDrive/results").mkdir(exist_ok=True)

#df.to_csv('/content/drive/MyDrive/my_output.csv')

"""node class for cantus firmus and first species counterpoint"""
class TreeNode():
    def __init__ (self,nodenote,accept):
        self.nodenote = nodenote
        self.accept = accept
        self.parent = None
        self.leapCount = 0

"""a standalone class to produce tree of cantus firmus based on fux fixed rules
rules will all be methods of cfproducer class"""
from music21 import *
import numpy as np
from graphviz import Digraph
import random
import json

class CFProducer():
    def __init__(self,epi,Mintervals):
        self.every_possible_interval = epi
        self.major_intervals = Mintervals
        self.n = 0
        self.root = None
        self.tree = None
        self.leaves = []

    def reset(self):
        self.n = 0
        self.root = None
        self.tree = None
        self.leaves = []

    def produceCF(self,n,tonic,filename="generated_melodies",verbose=False):
        """create and show cantus firmus
        return list of notes"""

        if verbose:
            print(f"generating all cantus firmus of length {n}")
        self.n = n

        #init start of tree
        self.root = TreeNode("N/A",False)

        self.generateTree(self.root,self.n,0,tonic,tonic,2,"N/A","N/A") #start climaxCount at 2 since tonic cant be climax anyways




        if verbose:

            cfstream = stream.Stream()
            for nnote in cf:
                cfstream.append(note.Note(nnote))
            cfstream.show()
        self.writeData(f"{filename}.txt")

    def getNewClimax(self,newNote,currClimax,climaxCount):
        #check if climax is duplicated for each note
        if newNote > currClimax:
            currClimax = newNote
            climaxCount = 1
        elif newNote == currClimax:
            climaxCount += 1

        return currClimax,climaxCount

    def getNextDirJumped(self,parent,nextNote,currDirJumped):
        """returns nextDirJumped for a given nextNote, ran on each possible note at each step

        dirJumped: the direction of a leap made to get to current note (accounts for arpeggios)
        dirJumped key is as follows:
        0 is no jump to next note
        1 is jump up besides p4
        -1 is jump down besides p4
        2 is perfect fourth up
        -2 is perfect fourth down
        3 is perfect fourth then a third up
        -3 is perfect fourth then a third down

        @param
        parent: current note node
        nextNote: the next note of melody, midi pitch
        currDirJumped: the dirJumped var for the note before current note to current note

        returns: nextDirJumped
        """
        nextDirJumped = 0
        sm = 0
        if parent.nodenote != "N/A":
            sm = nextNote - parent.nodenote
            if sm > 4: #leap up
                nextDirJumped = 1
                if sm == 5:
                    nextDirJumped = 2 #2 signifies this jump is a perfect fourth up
            elif sm < -4: #leap down
                nextDirJumped = -1
                if sm == -5:
                    nextDirJumped = -2
            #new variable to keep track of arpeggios: we can assume there is only 1 4:3:3 pattern in an arpeggio at most
            # due to octave and 3rd range, so we only need to keep track of if a 4th leap was made once (3rds arent leaps)
            #only valid 4th is perfect: 5 semitones
            if currDirJumped == 2 and sm in [3,4]: #if that last jump was a fourth and this move gonna be a 3rd in same direction (arpeggio)
                nextDirJumped = 3
            elif currDirJumped == -2 and sm in [-3,-4]: #down case
                nextDirJumped = -3
            elif currDirJumped == 3 and sm in [3,4]: #4:3:3 arpeggio has just transpired, need step down
                nextDirJumped = 1
            elif currDirJumped == -3 and sm in [-3,-4]: #down case (step up)
                nextDirJumped = -1

        return nextDirJumped

    def computeNewExtremes(self,lowestNote,highestNote,newNote):
        """compute extremes and give them back to generate tree to remember"""
        if lowestNote == "N/A" or highestNote == "N/A":
            lowestNote = newNote
            highestNote = newNote

        if lowestNote > newNote:
            lowestNote = newNote
        if highestNote < newNote:
            highestNote = newNote

        return lowestNote, highestNote

    def generateTree(self,parent,nodesLeft,dirJumped,tonic,currClimax,climaxCount,lowestNote,highestNote):
        """recursively generates tree of cantus firmuses

        @param parent node
        nodesLeft
        dirJumped
        tonic midi pitch num
        currClimax midi pitch num
        climaxCount
        lowestNote
        highestNote"""
        #find possible notes
        possibleNotes = self.getPossibleNotes(parent.nodenote,dirJumped,tonic,True,nodesLeft,lowestNote,highestNote)
        #add each note onto tree as node

        for n in possibleNotes:
            #get these climax variables for each possible note
            nClimax, nClimaxCount = self.getNewClimax(n,currClimax,climaxCount)

            if nodesLeft <= 1: #base case, final note
                #check that the climax isn't duplicated
                if nClimaxCount > 1:
                    continue #skip making an accepting node if it will duplicate the climax

                newNode = TreeNode(n,True)
                newNode.parent = parent
                newNode.leapCount = parent.leapCount #update leapcount
                self.leaves.append(newNode)
            else: #continue tree at next step
                newNode = TreeNode(n,False)
                newNode.parent = parent
                #calculate if the chosen note n is more than a third, and update dirJumped accordingly
                nextDirJumped = self.getNextDirJumped(parent,n,dirJumped)
                #update leap count in each node
                if abs(nextDirJumped) == 1:
                    newNode.leapCount = parent.leapCount + 1
                else:
                    newNode.leapCount = parent.leapCount
                #get the extremes for range calc
                newLowestNote,newHighestNote = self.computeNewExtremes(lowestNote,highestNote,n)

                self.generateTree(newNode,nodesLeft-1,nextDirJumped,tonic,nClimax,nClimaxCount,newLowestNote,newHighestNote)

    def partialIdentityMatrix(self,preserved_indeces):
        """creates an identity matrix of size of every possible interval,
        with only the specified diagonals remaining

        @param preserved_indices indices to keep"""
        matrixWidth = len(self.every_possible_interval)
        newMatrix = np.zeros((matrixWidth,matrixWidth),dtype =int)
        for i in preserved_indeces:
            newMatrix[i,i] = 1
        return newMatrix
    def partialIdentityMatrixDelete(self,deleted_indices):
        """creates an identity matrix of size of every possible interval,
        with only the specified diagonals as 0

        @param preserved_indices indices to delete"""
        matrixWidth = len(self.every_possible_interval)
        newMatrix = np.zeros((matrixWidth,matrixWidth),dtype =int)
        for i in range(len(self.every_possible_interval)):
            if i not in deleted_indices:
                newMatrix[i,i] = 1
        return newMatrix

    def EnsureCadence(self,weights,currentNote,tonic,nodesLeft,transposeOk):
        """ensures the  cadence happens

        @param weights
        currentNote
        tonic
        nodesLeft
        transposeOk: bool, whether landing on the tonic transposed up or down an octave is ok
        """
        possibleIntervals = []
        acceptableIntervals = []
        # allow it to end on cf tonic transposed up 1 octave or 2 octaves

        if nodesLeft == 2:
            # find intervals from currentfsnote to 2 semitones above transtonic and 1 semitone below (scale degree 2 and scale degree 7 major)
            # regardless of minor or major scale, these are the only two notes for cadence
            #multiply those times 1, else by 0
            if transposeOk:
                cadenceBeginnings = [tonic + 2,tonic - 1,tonic - 10,tonic + 11,tonic + 12 + 2,tonic - 12 - 1] #any other 2 or 7 is out of range
            else:
                cadenceBeginnings = [tonic + 2,tonic - 1] #any other 2 or 7 is out of range


            #get intervals from current note to 4 possible notes right before tonic (transtonic and transtonic transposed 1 octave higher)
            for cbnote in cadenceBeginnings:
                acceptableIntervals.append(cbnote - currentNote)

            for itvl in acceptableIntervals:
                if abs(itvl) <= 12: #to prevent leaps greater than 12 semitones (not possible in current framework)
                    possibleIntervals.append(itvl+12)
            return weights @ self.partialIdentityMatrix(possibleIntervals)



        elif nodesLeft == 1:
            #find interval from currentfsnote to transtonic
            #multilply that times 1 else by 0
            if transposeOk:
                acceptableIntervals = [tonic - currentNote,tonic +12 - currentNote,tonic - 12 - currentNote]
            else:
                acceptableIntervals = [tonic - currentNote]
            for itvl in acceptableIntervals:
                if abs(itvl) <= 12: #to prevent leaps greater than 12 semitones (not possible in current framework)
                    possibleIntervals.append(itvl+12)
            return weights @ self.partialIdentityMatrix(possibleIntervals)

        else:
            return weights

    def get_scale_degree_major(self,pitch, tonic):
        """
        pitch: int (MIDI number)
        tonic: int (MIDI number)

        returns: 1-7 if in major scale, else None
        """
        semitone = (pitch - tonic) % 12

        mapping = {
            0: 1,
            2: 2,
            4: 3,
            5: 4,
            7: 5,
            9: 6,
            11: 7
        }

        return mapping.get(semitone, None)

    def LimitToMajorScale(self,weights,tonic,currNote):
        """sets all intervals to 0 except for those in the given scale, depending on scaledegree

        @param prob the probability distribution we are working with
        @param tonic tonic we are using music21note
        @param currNote the current note were on music21 note

        @return list of floats between 0 and 1, len 25"""
        #first figure out scale degree
        scaleDegree = self.get_scale_degree_major(currNote,tonic)

        #use dictionary for scale
        AllowedIntervals = self.major_intervals[scaleDegree]

        for i in range(len(self.every_possible_interval)):
            weights[i] *= 1 if self.every_possible_interval[i] in AllowedIntervals else 0

        return weights

    def ResolveLeadingTone(self,weights,tonic,currNote):
        """if the current note is the leading tone, select only m2 as a possible interval

        @param weights the probability distribution we are working with
        @param tonic tonic we are using music21 note
        @param currNote the current note were on music21 note

        @return weights"""
        scaleDegree = self.get_scale_degree_major(currNote,tonic)

        if (scaleDegree == 7):
            return weights @ self.partialIdentityMatrix([13])
        else:
            return weights


    def LimitToRangeDynamic(self,weights,currNote,lowestNote,highestNote):
        """a new limit to range function that limits the entire interval of used notes to 1 and a 3rd,
        updating the possible region based on 1.3 above lowest and 1.3 below highest

        @param weights weights
        @param tonic tonic (text)
        @param currNote current note (music21 note)
        @param lowestNote lowestNote in cf so far (music21 note )
        @param highestNote highestNote in cf so far (music21 note)

        @return new weights"""

        #compute the bounds and update weights as such---
        unallowedIntervals = []
        #find highest allowed note which is 1.3 above lowest note
        highestValidNote = lowestNote + 12 + 4 #M10 is octave plus major 3rd, max range for cf we decided on
        #find interval from currNote to highest allowed note
        currToHighItvl = highestValidNote - currNote
        #find that interval in epi
        if abs(currToHighItvl) <= 12:
            epi_index = currToHighItvl+12
            #add anything above that interval to unallowed intervals
            unallowedIntervals.extend(range(epi_index+1,25))#stop exclusive
        #now same for  lowest allowed note
        lowestValidNote = highestNote - 12 -4

        currToLowItvl = lowestValidNote - currNote

        if abs(currToLowItvl) <= 12:
            epi_index = currToLowItvl+12
            #add anything below that interval to unallowed intervals
            unallowedIntervals.extend(range(epi_index)) #start inclusive

        return weights @ self.partialIdentityMatrixDelete(unallowedIntervals)

    def getPossibleNotes(self,currentNote,dirJumped,tonic,major,nodesLeft,lowestNote,highestNote):
        """returns list of possible notes (music21 notes)

        params:
        currentNote: current music21 note of cantus firmus
        dirJumped: the direction of a leap made to get to current note (0 is no jump, 1 is jump up, -1 is jump down)
            extended - (2 is perfect fourth up, -2 is perfect fourth down, 3 is perfect fourth then a third up, -3 is perfect fourth then a third down)
        tonic: music21 note of tonic
        major: boolean whether the cf is major or not
        nodesLeft: number of remaining notes in cf (1 is the lowest you can expect)
        lowestNote: lowest music21 note in melody so far
        highestNote: highest music21 note in melody so far

        returns:
        list of possible notes (music21 notes)
        """
        #check first case
        if(currentNote == "N/A"): #if no notes have been laid yet, possible notes are tonic
            return [tonic]

        weights=[1]*len(self.every_possible_interval)

        #good melody filters (CF)
        #1) limit to key
        if (major):
            #limit intervals to intervals in scale
            weights = self.LimitToMajorScale(weights,tonic,currentNote)
        else:
            pass #TODO
        #2) limit to range
        weights = self.LimitToRangeDynamic(weights,currentNote,lowestNote,highestNote)
        #3) step back after leap
        if (dirJumped > 0):
            possibleStepsAfterLeapUp = [10,11]#stepwise down #ONLY TWOS
            #if dirJumped is 1, only allow step down, if dirJumped is 2 allow step down or 3rd up (minor or major depending on scale degree), same for dirJumped 3
            if dirJumped > 1:
                #find scale degree to decide between minor or major required to stay in key
                possibleStepsAfterLeapUp.extend([15,16]) #add both minor and major, limitToKey already handles eliminating the wrong one(if its already 0, itll stay 0)
            #get new weights filtred for step backs
            weights = weights @ self.partialIdentityMatrix(possibleStepsAfterLeapUp)
        elif (dirJumped < 0):
            possibleStepsAfterLeapDown = [13,14]#stepwise up #ONLY TWOS

            if dirJumped < -1:
                #find scale degree to decide between minor or major required to stay in key
                possibleStepsAfterLeapDown.extend([8,9])
            #get new weights
            weights = weights @ self.partialIdentityMatrix(possibleStepsAfterLeapDown)
        #4) end in cadence
        weights = self.EnsureCadence(weights,currentNote,tonic,nodesLeft,False) #for cantus firmus, transpose not Ok: must end on untransposed tonic
        #5) resolve leading tone always
        weights = self.ResolveLeadingTone(weights,tonic,currentNote)
        #disallow oblique motion no matter what in cantus firmus
        weights = weights @ self.partialIdentityMatrixDelete([12])#12 is middle, P1

        #convert interval weights to notes
        possibleNotes = []
        for i in range(len(weights)):
            if weights[i] != 0:
                associatedInterval = self.every_possible_interval[i]
                possibleNotes.append(currentNote + associatedInterval)
        return possibleNotes

    def writeData(self,filename):
        print(f"There are {len(self.leaves)} possible cantus firmuses of length {self.n}")
        lines = []

        for leafNode in self.leaves:
            notes = [note.Note(leafNode.nodenote).nameWithOctave]
            currnode = leafNode
            for _ in range(self.n-1):
                currnode = currnode.parent
                notes.append(note.Note(currnode.nodenote).nameWithOctave)

            writeDict = {
                "melody": ",".join(reversed(notes)),
                "leapCount": leafNode.leapCount
            }
            lines.append(json.dumps(writeDict))
        with open(filename,"w") as f:
            f.write("\n".join(lines))

"""Class to produce all possible first species counterpoints on a given cantus firmus
Inherits from CFProducer"""
from music21 import *
import numpy as np
from graphviz import Digraph
import random
import json
import os
from pathlib import Path

#use classes to build a tree with all the possibilities, then follow random to choose one fscp


class FSProducer(CFProducer):
    def __init__(self,epi,Mintervals):
        self.every_possible_interval = epi
        self.major_intervals = Mintervals
        self.cflen =0
        self.root = None
        self.tree = None
        self.leaves = []
        self.tonic = None #music21 note

    def reset(self):
        """resets all variables to avoid leakage across different runs with same fsproducer instance"""
        self.cflen = 0
        self.root = None
        self.tree = None
        self.leaves = []
        self.tonic = None

    def produceFS(self,cf, verbose =False):
        """given the cantus firmus, produce a first species counterpoint melody that lines up with
        counterpoint rules and print both melodies

        for now assume quarterLength is always 1 when creating notes (default)

        @params
        list cf a list of music21 notes corresponding to the cantus firmus"""

        if verbose:
            print(cf)
            print("\nlength: ",len(cf),"\n")

        #start with blank node (because there are multiple possible starting notes (1  3 and 5))
        self.root = TreeNode("N/A",False)
        #get length of cf
        cflen = len(cf)
        self.cflen =cflen
        self.tonic = cf[0]
        #insert dummy at start of cf for generating tree
        cf.insert(0,"N/A")
        #create tree
        self.generateFSTree(self.root,cflen,cf,0,self.tonic,1,"N/A","N/A",False)

        #make sure to already have a results folder
        #if not os.path.isdir("results"):
            #Path("results").mkdir(exist_ok=True)
        self.writeData("/content/drive/MyDrive/results/" + self.convertCFtoFilename(cf[1:]))

    def generateFSTree(self,parent, nodesLeft,cf,dirJumped,currClimax,climaxCount,lowestNote,highestNote,tieUsed):
        """generate all possible first species to accompany given cantus firmus, abandoning paths as they fail.
        final, correct paths will have a value of True as their accepting parameter, indicating the path to get
        to that node was a valid counterpoint

        @params
        parent parent of currnode
        nodesLeft how many nodes until first species is finished
        cf original cantus firmus
        dirJumped
        currClimax music21 note
        climaxCount
        lowestNote
        highestNote
        tieUsed bool"""
        #find possible notes
        possibleNotes = self.getPossibleNotes(parent.nodenote,cf[-(nodesLeft+1)],cf[-nodesLeft],dirJumped,cf[1] + 12,True,nodesLeft,lowestNote,highestNote,tieUsed)
        #add each note onto tree as node
        for n in possibleNotes:
            #get these climax variables for each possible note
            nClimax, nClimaxCount = self.getNewClimax(n,currClimax,climaxCount)

            if nodesLeft <= 1: #base case, final note
                if nClimaxCount > 1:
                    continue #skip making an accepting node if it will duplicate the climax

                newNode = TreeNode(n,True)
                newNode.parent = parent
                newNode.tieUsed = tieUsed
                newNode.leapCount = parent.leapCount #update leapcount
                self.leaves.append(newNode)
            else: #continue tree at next step
                newNode = TreeNode(n,False)
                newNode.parent = parent
                #calculate if the chosen note n is more than a third, and update dirJumped accordingly
                nextDirJumped = self.getNextDirJumped(parent,n,dirJumped)

                if abs(nextDirJumped) == 1:
                    newNode.leapCount = parent.leapCount + 1
                else:
                    newNode.leapCount = parent.leapCount

                newLowestNote,newHighestNote = self.computeNewExtremes(lowestNote,highestNote,n)

                #ensureCadence stops last note from being a tie
                if n == parent.nodenote:
                    tieUsed = True

                self.generateFSTree(newNode,nodesLeft-1,cf,nextDirJumped,nClimax,nClimaxCount,newLowestNote,newHighestNote,tieUsed)

    def EnsureOppositeCadence(self,weights,currentFSnote, transtonic,nodesLeft,nextCFnote):
        """ensures that the first species counterpoint has the opposite cadence from the cantus firmus"""
        if nodesLeft == 2:
            acceptableIntervals = []
            cadenceBeginnings = []
            possibleIntervals = []
            #if there are two notes left, and nextcfnote is 2nd degree we need 7th for fs
            #and vice versa
            #find if nextCFnote is second degrre or seventh
            scaleDegree = self.get_scale_degree_major(nextCFnote,transtonic)
            if scaleDegree == 2:
                #cf cadence is 2nd degree to tonic
                cadenceBeginnings =[transtonic - 1,transtonic + 11,transtonic - 12 - 1]
                for cbnote in cadenceBeginnings:
                    acceptableIntervals.append(cbnote - currentFSnote)
                for itvl in acceptableIntervals:
                    if abs(itvl) <= 12 and abs(itvl) >= -12: #to prevent leaps greater than 12 semitones (not possible in current framework)
                        possibleIntervals.append(itvl+12)
            elif scaleDegree == 7:
                #cf cadence is 7th degree to tonic
                cadenceBeginnings = [transtonic + 2,transtonic + 12 + 2,transtonic - 10]
                for cbnote in cadenceBeginnings:
                    acceptableIntervals.append(cbnote - currentFSnote)
                for itvl in acceptableIntervals:
                    if abs(itvl) <= 12 and abs(itvl) >= -12: #to prevent leaps greater than 12 semitones (not possible in current framework)
                        possibleIntervals.append(itvl+12)
            else:
                print("There is no cadence in cantus firmus")

            return weights @ self.partialIdentityMatrix(possibleIntervals)


        return weights

    def AvoidParallelPerfectConsonance(self,weights,currentCFnote,nextCFnote, currentFSnote):

        verticalinterval = currentFSnote - currentCFnote

        #if interval between current cf and fs is perfect
        if((abs(verticalinterval)%12) in [7,0]):
            #then find interval from current cf to next cf
            horintervalcf = nextCFnote - currentCFnote

            #this interval is not allowed
            return weights @ self.partialIdentityMatrixDelete([horintervalcf+12])
        else:

            return weights

    def AvoidSameConsecutivePerfect(self,weights,currentCFnote,nextCFnote,currentFSnote):

        verticalinterval = currentFSnote - currentCFnote
        currentperfect = (abs(verticalinterval)%12)
        if(currentperfect in [7,0]):
            #loop through each weight thats not already 0
            for i in range(len(weights)):
                if weights[i] != 0: #theres no point calculating it on stuff thats already 0
                    nextFSnote = currentFSnote + self.every_possible_interval[i]
                    nextvertinterval = nextFSnote - nextCFnote
                    if (abs(nextvertinterval)%12) == currentperfect:
                        weights[i] = 0 #=0, *= 0,  weights @ self.partialIdentityMatrixDelete([i]) all do same thing
        return weights



    def AvoidDirectPerfectIntervals(self,weights,currentCFnote,nextCFnote, currentFSnote):
        unallowedIntervals = []
        #for every possible next fs interval,
        for i in range(len(weights)):
            if weights[i] != 0: #theres no point calculating it on stuff thats already 0
                #if the fs interval is not by step ( just 2 now)
                if(i not in [10,11,13,14]):
                    #and if the vertical interval between that resulting note and next cfnote is perfect,
                    vertinterval = (currentFSnote + self.every_possible_interval[i]) - nextCFnote
                    if ((abs(vertinterval)%12) in [7,0]):
                        #and if the interval is similar to cf interval
                        #cf interval polarity
                        cfinterval = nextCFnote - currentCFnote
                        cfintervalpolarity = cfinterval/abs(cfinterval) #divide by 0 error cant occur because there is no tie in cf
                        #fs interval polarity
                        fsintervalpolarity = 0
                        if self.every_possible_interval[i] < 0:
                            fsintervalpolarity = -1
                        elif self.every_possible_interval[i] > 0:
                            fsintervalpolarity = 1
                        #if interval is P1, leave it at 0
                        else:
                            fsintervalpolarity = 1

                        if (fsintervalpolarity == cfintervalpolarity):
                            #then delete it
                            unallowedIntervals.append(i) #i is already in terms of every possible interval (no need to add 12)
        return weights @ self.partialIdentityMatrixDelete(unallowedIntervals)

    def NoSimultaneousLeaps(self,weights,currentCFnote,nextCFnote):

        cfinterval = nextCFnote - currentCFnote
        if(abs(cfinterval) > 4):
            return weights @ self.partialIdentityMatrix([8,9,10,11,12,13,14,15,16])
        return weights

    def LimitToConsonantVertical(self,weights,currentFSnote,nextCFnote):
        """limits to consonant vertical intervals between cf and fs

        @param weights
        """
        #for each interval, calculate the note you would end up at from current fs note


        for i in range(len(weights)):
            if weights[i] != 0: #theres no point calculating it on stuff thats already 0
                associatedInterval = self.every_possible_interval[i]
                newnote = currentFSnote + associatedInterval

                # then calculate if the interval between nextcfnote and that note is consonant
                newInterval = newnote - nextCFnote

                if (abs(newInterval) % 12) not in [3,4,7,8,9,0]:
                    weights[i] *= 0
        return weights

    def NoOverlap(self,weights,nextCFnote,currentFSnote,nodesLeft):
        """ensures no overlap between cantus firmus and fs"""
        if not nodesLeft == 1:#allows last note to overlap
            for i in range(len(weights)):
                if weights[i] != 0: #theres no point calculating it on stuff thats already 0
                    associatedInterval = self.every_possible_interval[i]
                    newnote = currentFSnote + associatedInterval

                    #if the new note is equal to or lower than cf note, then disqualify it
                    if (newnote <= nextCFnote):
                        weights[i] *= 0
        return weights


    def getPossibleNotes(self,currentFSnote,currentCFnote,nextCFnote,dirJumped,transtonic,major,nodesLeft,lowestNote,highestNote,tieUsed):
        """returns all possible notes by eliminating intervals from currentFSnote

        @params
        currentFSnote string representation of current fsnote (or "N/A")
        currentCFnote music 21 note (or "N/A")
        nextCFnote music 21 note (or "N/A")
        dirJumped indicates direction jumped last interval in fs (>0 means up, <0 means down, 0 means step)
        transtonic the transtonic of the scale, as music21 note, trasposed up an octave
        major whether the cantus firmus in the major scale or not (bool)
        nodesLeft how many nodes/notes are left in the song
        lowestNote: lowest music21 note in melody so far
        highestNote: highest music21 note in melody so far
        tieUsed: whether a tie has been used in the melody yet

        @return
        list of possible notes (music 21 notes)
        """

        #check first case
        if(currentFSnote == "N/A"): #if no notes have been laid yet, possible notes are transtonic, third, and fifth
            return [transtonic,transtonic + 4,transtonic + 7] #this is all up an octave( keeps distance from cf to maximize produced fs while not duplicating melodies by having both transposed triads)
            #return [transtonic,transtonic.transpose("-P8"),transtonic.transpose("-m6"),transtonic.transpose("-P4")] #does tonic triad + octave

        weights=[1]*len(self.every_possible_interval)


        #good melody filters (CF)
        #1) limit to key
        if (major):
            #limit intervals to intervals in scale
            weights = self.LimitToMajorScale(weights,transtonic,currentFSnote)
        else:
            pass #TODO
        #2) limit to range
        weights = self.LimitToRangeDynamic(weights,currentFSnote,lowestNote,highestNote)
        #3) step back after leap
        if (dirJumped > 0):
            possibleStepsAfterLeapUp = [10,11]#stepwise down #TWOS ONLY
            #if dirJumped is 1, only allow step down, if dirJumped is 2 allow step down or 3rd up (minor or major depending on scale degree), same for dirJumped 3
            if dirJumped > 1:
                #find scale degree to decide between minor or major required to stay in key
                possibleStepsAfterLeapUp.extend([15,16]) #add both minor and major, limitToKey already handles eliminating the wrong one(if its already 0, itll stay 0)
            #get new weights filtred for step backs
            weights = weights @ self.partialIdentityMatrix(possibleStepsAfterLeapUp)
        elif (dirJumped < 0):
            possibleStepsAfterLeapDown = [13,14]#stepwise up #ONLY TWOS

            if dirJumped < -1:
                #find scale degree to decide between minor or major required to stay in key
                possibleStepsAfterLeapDown.extend([8,9])
            #get new weights
            weights = weights @ self.partialIdentityMatrix(possibleStepsAfterLeapDown)
        #4) end in cadence
        weights = self.EnsureCadence(weights,currentFSnote,transtonic,nodesLeft,True) #ok to end on transposed tonic

        #DO NOT have to resolve leading tone for first species-- weights = self.ResolveLeadingTone(weights,transtonic,currentFSnote)

        #first species exclusive filters
        #1) only consonant vertical intervals NO second, fourth, seventh, aug, dim, tritone
        weights = self.LimitToConsonantVertical(weights,currentFSnote,nextCFnote)
        #2) no parallel perfect consonances
        weights = self.AvoidParallelPerfectConsonance(weights,currentCFnote,nextCFnote,currentFSnote)
        #2.5) no contrary perfect intervals (accomplished by avoid same consecutive perfect interval)
        weights = self.AvoidSameConsecutivePerfect(weights,currentCFnote,nextCFnote,currentFSnote)
        #3) no direct perfect intervals
        weights = self.AvoidDirectPerfectIntervals(weights,currentCFnote,nextCFnote,currentFSnote)
        #4) no simultaneous leaps
        weights = self.NoSimultaneousLeaps(weights,currentCFnote,nextCFnote)
        #4.5) voices may not overlap or cross
        weights = self.NoOverlap(weights,nextCFnote,currentFSnote,nodesLeft)
        #5)end on opposite cadence
        weights = self.EnsureOppositeCadence(weights,currentFSnote,transtonic,nodesLeft,nextCFnote)
        #once all filters have been applied------------------
        #6) disallow tied note if tie has been used
        if tieUsed:
            weights = weights @ self.partialIdentityMatrixDelete([12])#12 is middle, P1

        possibleNotes = []
        for i in range(len(weights)):
            if weights[i] != 0:
                associatedInterval = self.every_possible_interval[i]
                possibleNotes.append(currentFSnote + associatedInterval)

        return possibleNotes


    def convertCFtoFilename(self,cf):
        return "_".join(note.Note(midinote).nameWithOctave for midinote in cf)

    def writeData(self,filename):
        #print(f"There are {len(self.leaves)} possible first species counterpoins for given cantus firmus")
        with open(filename,"w") as f:
            for leafNode in self.leaves:
                notes = [note.Note(leafNode.nodenote).nameWithOctave]
                currnode = leafNode
                for _ in range(self.cflen-1):
                    currnode = currnode.parent
                    notes.append(note.Note(currnode.nodenote).nameWithOctave)

                #find starting note scale degree
                scaleDegree = self.get_scale_degree_major(note.Note(notes[-1]).pitch.midi,self.tonic)

                writeDict = {
                    "melody": ",".join(reversed(notes)),
                    "leapCount": leafNode.leapCount,
                    "tieUsed" : leafNode.tieUsed,
                    "startingNote": scaleDegree
                }
                f.write(json.dumps(writeDict) + "\n")


Mounted at /content/drive


In [ ]:
#rename results folder in google drive after each trial
if not os.path.isdir("/content/drive/MyDrive/results"):
    Path("/content/drive/MyDrive/results").mkdir(exist_ok=True)

import time
start = time.perf_counter()
LENGTH = 9

def melodyToMidi(melody):
    melodyNotes = melody.split(",")
    returnMidi = []
    for notename in melodyNotes:
        returnMidi.append(note.Note(notename).pitch.midi)
    return returnMidi

print(f"Collecting data for all cantus firmus of length {LENGTH}")


#every possible interval within an octave from a note
every_possible_interval = list(range(-12, 13))  # semitones
#possible intervals from each scale degree in a major scale (leaving out tritones and 7th intervals)
MajorIntervalsFull = {
    1:  [ 2, 4, 5, 7, 9, 12, -1, -3, -5, -7, -8, -12, 0],
    2:  [ 2, 3, 5, 7, 12, -2, -3, -5, -7, -9, -12, 0],
    3:  [ 1, 3, 5, 8, 12, -2, -4, -5, -7, -9, -12, 0],
    4:  [ 2, 4, 7, 9, 12, -1, -3, -5, -8, -12, 0],
    5:  [ 2, 4, 5, 7, 9, 12, -2, -3, -5, -7, -8, -12, 0],
    6:  [ 2, 3, 5, 7, 8, 12, -2, -4, -5, -7, -9, -12, 0],
    7:  [ 1, 3, 5, 8, 12, -2, -4, -7, -9, 0]
}
#cantus firmus
CFcomposer = CFProducer(every_possible_interval,MajorIntervalsFull)

CFcomposer.produceCF(LENGTH,60,verbose=False,filename=f"/content/drive/MyDrive/generated_melodies{LENGTH}")

#first species
FScomposer = FSProducer(every_possible_interval,MajorIntervalsFull)
with open(f"/content/drive/MyDrive/generated_melodies{LENGTH}.txt","r") as f:
    for line in f:
        cfdict = json.loads(line.strip())

        #convert cf dict melody to list of music21 notes
        cf = melodyToMidi(cfdict["melody"])
        #print(cf)
        FScomposer.reset()
        FScomposer.produceFS(cf,verbose=False)

end = time.perf_counter()
print(f"Elapsed time: {end - start:.6f} seconds")
from google.colab import output
# Replace the URL with any public audio file link
output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')

There are 8311 possible cantus firmuses of length 9


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/results/C4_C3_D3_B2_C3_D3_F3_D4_C4'

In [ ]:
"""Run this experiment to not generate new cf list but just use one already generated
this just add up everything already in results and skip to that line in generated melodies,
 so you can start where you left off if you ran 1 but closed it"""
#if not os.path.isdir("/content/drive/MyDrive/results"):
    #Path("/content/drive/MyDrive/results").mkdir(exist_ok=True)
#rename results folder in google drive after each trial
import time
start = time.perf_counter()

def melodyToMidi(melody):
    melodyNotes = melody.split(",")
    returnMidi = []
    for notename in melodyNotes:
        returnMidi.append(note.Note(notename).pitch.midi)
    return returnMidi

print("Collecting data for cantus firmuses in generated_melodies.txt")

#every possible interval within an octave from a note
every_possible_interval = list(range(-12, 13))  # semitones
#possible intervals from each scale degree in a major scale (leaving out tritones and 7th intervals)
MajorIntervalsFull = {
    1:  [ 2, 4, 5, 7, 9, 12, -1, -3, -5, -7, -8, -12, 0],
    2:  [ 2, 3, 5, 7, 12, -2, -3, -5, -7, -9, -12, 0],
    3:  [ 1, 3, 5, 8, 12, -2, -4, -5, -7, -9, -12, 0],
    4:  [ 2, 4, 7, 9, 12, -1, -3, -5, -8, -12, 0],
    5:  [ 2, 4, 5, 7, 9, 12, -2, -3, -5, -7, -8, -12, 0],
    6:  [ 2, 3, 5, 7, 8, 12, -2, -4, -5, -7, -9, -12, 0],
    7:  [ 1, 3, 5, 8, 12, -2, -4, -7, -9, 0]
}

file_count = sum(1 for p in Path("/content/drive/MyDrive/results").iterdir() if p.is_file())
print(file_count)

#first species
FScomposer = FSProducer(every_possible_interval,MajorIntervalsFull)
currLine = 0
with open("/content/drive/MyDrive/generated_melodies16S.txt","r") as f: #rename this file
    for line in f:

        currLine += 1
        if currLine <= file_count:
            continue

        cfdict = json.loads(line.strip())

        #convert cf dict melody to list of music21 notes
        cf = melodyToMidi(cfdict["melody"])
        #print(cf)
        FScomposer.reset()
        FScomposer.produceFS(cf,verbose=False)
end = time.perf_counter()
print(f"Elapsed time: {end - start:.6f} seconds")
from google.colab import output
# Replace the URL with any public audio file link
output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')

686
Elapsed time: 23566.974190 seconds


KeyboardInterrupt: 

In [ ]:
"""run this program to generate all first species counterpoints on a random sample of
specified size of cantus firmuses of specified length"""
if not os.path.isdir("/content/drive/MyDrive/results"):
    Path("/content/drive/MyDrive/results").mkdir(exist_ok=True)
#rename results folder in google drive after each trial
import time
start = time.perf_counter()
N = 1000 #number of cf to find all fs on
LENGTH = 13 #length of cf (and therefore fs)
SEED = 0


def melodyToMidi(melody):
    melodyNotes = melody.split(",")
    returnMidi = []
    for notename in melodyNotes:
        returnMidi.append(note.Note(notename).pitch.midi)
    return returnMidi

random.seed(SEED)

print(f"Collecting data for {N} random cantus firmus of length {LENGTH} on seed {SEED}")

#every possible interval within an octave from a note
every_possible_interval = list(range(-12, 13))  # semitones
#possible intervals from each scale degree in a major scale (leaving out tritones and 7th intervals)
MajorIntervalsFull = {
    1:  [ 2, 4, 5, 7, 9, 12, -1, -3, -5, -7, -8, -12, 0],
    2:  [ 2, 3, 5, 7, 12, -2, -3, -5, -7, -9, -12, 0],
    3:  [ 1, 3, 5, 8, 12, -2, -4, -5, -7, -9, -12, 0],
    4:  [ 2, 4, 7, 9, 12, -1, -3, -5, -8, -12, 0],
    5:  [ 2, 4, 5, 7, 9, 12, -2, -3, -5, -7, -8, -12, 0],
    6:  [ 2, 3, 5, 7, 8, 12, -2, -4, -5, -7, -9, -12, 0],
    7:  [ 1, 3, 5, 8, 12, -2, -4, -7, -9, 0]
}
#cantus firmus
CFcomposer = CFProducer(every_possible_interval,MajorIntervalsFull)

CFcomposer.produceCF(LENGTH,60,verbose=False,filename=f"/content/drive/MyDrive/generated_melodies{LENGTH}")

CFcomposer.reset()

#run CFproducer and make a new file with only a random sample from that, then run the experiment on that
#read all data in
with open(f"/content/drive/MyDrive/generated_melodies{LENGTH}.txt","r") as f:
    content = f.read()
#split content by \n
content = set(content.split("\n"))
with open(f"/content/drive/MyDrive/generated_melodies{LENGTH}S.txt","w") as f:
    for i in range(N):
        #get a random line from content and remove it
        line = random.choice(tuple(content))
        content.remove(line)
        f.write(line + "\n")
time.sleep(.01)
end = time.perf_counter()
print(f"Elapsed time: {end - start:.6f} seconds")

#first species
FScomposer = FSProducer(every_possible_interval,MajorIntervalsFull)
with open(f"/content/drive/MyDrive/generated_melodies{LENGTH}S.txt","r") as f:
    for line in f:
        if line.strip():
            cfdict = json.loads(line.strip())

            #convert cf dict melody to list of music21 notes
            cf = melodyToMidi(cfdict["melody"])
            #print(cf)
            FScomposer.reset()
            FScomposer.produceFS(cf,verbose=False)
end = time.perf_counter()
print(f"Elapsed time: {end - start:.6f} seconds")
from google.colab import output
# Replace the URL with any public audio file link
output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')

There are 2912744 possible cantus firmuses of length 13
Elapsed time: 1424.025532 seconds
Elapsed time: 8023.210220 seconds


In [ ]:
from google.colab import output
# Replace the URL with any public audio file link
output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')
